# PR #150 repricing experiments — five ideas on the Algorithm 34 surface

**Context (2026-07-25).** flopscope PR #150 (+ #151 follow-ups, merged, v0.9.1) is live on the
grader. Our leaderboard best reverted to **submission 315516** (July 9, original Algorithm 17:
Strassen-accelerated dense propagation, gather-free, adaptive N). The Algorithm 34 ship surface
(317421) leans on `take`-gather packing that the repricing quadrupled, and — measured below —
runs its dominant matmuls in **float64**, which v0.9.1 bills at 2× the float32 rate — on top of
the 4/elem `take` gathers, enough to push the ship bytes over budget.

**What this notebook tests** (each section carries its hypothesis block):

0. **Dtype hygiene** — cast `mlp.weights` float64→float32 before any op.
1. **Idea 1** — algo34's fixed-N + antithetic pilots + guarded 2-block Strassen with the packed
   path disabled (315516's routing philosophy on the 317421 sampling surface).
2. **Idea 2** — the layer-30/31 on-neuron fold: disable it, thread on-neurons as identity
   earlier, and tune the on-threshold that decides how much goes linear.
3. **Idea 3** — drop the 5% pilot stage; classify analytic probe candidates with the 20% stage
   directly.
4. **Idea 4** — dead-neuron reporting: analytic value vs zero vs scaled blend (post-hoc γ sweep).
5. **Idea 5** — re-capture activation sparsity under the NEW pricing: recalibrated pack/dense
   routing from measured 0.9.1 op prices.

**Protocol.** Public mini split (baked N=1e9 ground truth), fixed net ids, `estimator` =
`exp_pr150_variants.py` (bit-identical to algo34 at default config — verified max|Δ|=0 vs the
317421 prediction cache). Billing = the **merged flopscope v0.9.1** installed in
`.flopscope-pr150/`. Score proxy = `final_layer_mse × max(0.1, F/B)` with **FLOP-only**
multiplier; over-budget nets are scored grader-true (zeroed predictions, multiplier 1.0).
Residual wall-time is recorded but excluded (laptop contention; grader λ·R differs). Final
validation of any winner goes through `whest run` + a real submission.

In [1]:
import sys, json, time, hashlib
from pathlib import Path

REPO = Path("/Users/natashastewart/whest-kprop-experiments")
# NOTE (2026-07-25, post-execution edit): the isolated .flopscope-pr150 build
# was removed once the merged flopscope 0.9.1 / whestbench 0.13.0 were pinned
# in pyproject — the project venv now bills identically, so no path insert.
sys.path.insert(0, str(REPO))

import numpy as np
import flopscope as flops
import flopscope.numpy as fnp
import whestbench as wb
from whestbench import SetupContext
from flopscope import _weights as fw
import exp_pr150_variants as ev

assert flops.__version__.startswith("0.9.1"), flops.__version__
assert fw.get_weight("take") == 4.0, "PR150 pricing not active"
flops.configure(symmetry_warnings=False, callback_warnings=False)

BUDGET = 272_000_000_000
ds = wb.load_dataset("aicrowd/arc-whestbench-public-2026", revision="v1-phase1", split="mini")
GT = np.asarray(ds["all_layer_means"], np.float64)
FULL_IDS = list(range(0, 100, 10))   # 10 nets
SWEEP_IDS = FULL_IDS[:6]             # cheaper grids
ZERO_MSE = {i: float((GT[i][-1] ** 2).mean()) for i in range(GT.shape[0])}
CACHE = REPO / ".exp_pr150_cache"
CACHE.mkdir(exist_ok=True)

def _cfg_hash(overrides):
    return hashlib.md5(json.dumps({k: repr(v) for k, v in sorted(overrides.items())}).encode()).hexdigest()[:10]

def run_variant(tag, overrides, ids, capture=False):
    """Run (or load cached) predicts for one config. Returns list of per-net dicts."""
    rows, h = [], _cfg_hash(overrides)
    for i in ids:
        f = CACHE / f"{tag}__net{i}.npz"
        need_capture = capture and not (f.exists() and "cap_final_row" in np.load(f, allow_pickle=True))
        if f.exists() and not need_capture:
            z = np.load(f, allow_pickle=True)
            assert str(z["cfg_hash"]) == h, f"stale cache {f}: config changed, delete it"
            pred = z["pred"].astype(np.float64)
            F, resid, wall = float(z["F"]), float(z["resid"]), float(z["wall"])
            ops = json.loads(str(z["ops"]))
            cap = {k[4:]: z[k] for k in z.files if k.startswith("cap_")}
        else:
            ev.reset_config(); ev.configure(**overrides)
            if capture:
                ev.CAPTURE.clear(); ev.CAPTURE["on"] = True
            E = ev.Estimator()
            E.setup(SetupContext(seed=0, width=256, depth=32, flop_budget=BUDGET,
                                 api_version="v1", submission_dir=str(REPO)))
            m = wb.mlp_at(ds, i)
            flops.budget_reset()
            t0 = time.time()
            with flops.BudgetContext(flop_budget=40 * BUDGET) as b:
                pred = np.asarray(E.predict(m, BUDGET), np.float64)
            wall = time.time() - t0
            d = flops.budget_summary_dict(b)
            F, resid = float(d["flops_used"]), float(d["residual_wall_time_s"])
            ops = {k: v["flop_cost"] for k, v in d["operations"].items()}
            cap = {}
            if capture:
                cap = {k: np.asarray(v) for k, v in ev.CAPTURE.items() if k != "on"}
                ev.CAPTURE.clear()
            np.savez_compressed(f, pred=pred.astype(np.float32), F=F, resid=resid, wall=wall,
                                cfg_hash=h, ops=json.dumps(ops),
                                **{f"cap_{k}": v for k, v in cap.items()})
        mse = float(((pred[-1] - GT[i][-1]) ** 2).mean())
        mult = max(0.1, F / BUDGET)
        over = F > BUDGET
        rows.append(dict(net=i, F=F, mult=mult, over=over, mse=mse,
                         adj_ideal=mse * mult,
                         adj_grader=(ZERO_MSE[i] if over else mse * mult),
                         resid=resid, wall=wall, ops=ops, pred=pred, cap=cap))
    return rows

def summ(rows):
    F = np.mean([r["F"] for r in rows]); n_over = sum(r["over"] for r in rows)
    return dict(F_G=F / 1e9, mult=np.mean([r["mult"] for r in rows]), n_over=n_over,
                raw=np.mean([r["mse"] for r in rows]),
                adj=np.mean([r["adj_grader"] for r in rows]),
                adj_ideal=np.mean([r["adj_ideal"] for r in rows]),
                worst=max(r["adj_grader"] for r in rows),
                resid_s=np.mean([r["resid"] for r in rows]))

def table(results, baseline=None):
    hdr = f"{'variant':26s} {'F(G)':>8s} {'mult':>6s} {'#over':>5s} {'raw MSE':>10s} {'adj(grader)':>11s} {'adj(ideal)':>10s} {'Δadj%':>7s}"
    print(hdr); print("-" * len(hdr))
    base = summ(results[baseline]) if baseline else None
    for name, rows in results.items():
        s = summ(rows)
        d = f"{100 * (s['adj'] / base['adj'] - 1):+6.1f}%" if base and name != baseline else "      "
        print(f"{name:26s} {s['F_G']:8.1f} {s['mult']:6.3f} {s['n_over']:5d} "
              f"{s['raw']:10.3e} {s['adj']:11.3e} {s['adj_ideal']:10.3e} {d}")

print(f"harness ready: {len(FULL_IDS)} nets, budget {BUDGET/1e9:.0f}G, flopscope {flops.__version__}")

/Users/natashastewart/whest-kprop-experiments/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/z7/gqv5ccjs5r723sth_ds6lrwm0000gn/T/ipykernel_60843/3138760293.py:19: ConfigureNoOpWarning: flops.configure() configures the in-process flopscope backend only; it is a no-op on flopscope-client and the evaluation servers, so these settings will not affect a graded submission.
  flops.configure(symmetry_warnings=False, callback_warnings=False)
whestbench: downloading 'mini' split from hf://aicrowd/arc-whestbench-public-2026@v1-phase1


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 60567.57it/s]

harness ready: 10 nets, budget 272G, flopscope 0.9.1+np2.2.6


## 0. Measured v0.9.1 op prices (the new cost model in numbers)

Everything below is *measured* against the merged build, not inferred from the PR text. The
derived pack-vs-dense crossover feeds idea 5.

In [2]:
def _price(fn):
    flops.budget_reset()
    with flops.BudgetContext(flop_budget=10**13) as b:
        fn()
    d = flops.budget_summary_dict(b)
    return {k: v["flop_cost"] for k, v in d["operations"].items()}

rng = np.random.default_rng(0)
Xf32 = rng.standard_normal((4096, 256)).astype(np.float32)
Wf32 = rng.standard_normal((256, 256)).astype(np.float32)
Wf64 = Wf32.astype(np.float64)
MASK = (Xf32 > 0)
IDX = rng.integers(0, 256, size=(4096, 64))

mm = 2 * 4096 * 256 * 256
rows = []
rows.append(("matmul f32@f32 (per 2mkn)", _price(lambda: fnp.array(Xf32) @ fnp.array(Wf32))["matmul"] / mm))
rows.append(("matmul f32@f64 (per 2mkn)", _price(lambda: fnp.array(Xf32) @ fnp.array(Wf64))["matmul"] / mm))
rows.append(("matmul f64@f64 (per 2mkn)", _price(lambda: fnp.array(Xf32.astype(np.float64)) @ fnp.array(Wf64))["matmul"] / mm))
rows.append(("take rows w[order] (per out elem)", _price(lambda: fnp.take(fnp.array(Wf32), fnp.array(IDX), axis=0))["take"] / IDX.size / 256))
rows.append(("take_along_axis (per out elem)", _price(lambda: fnp.take_along_axis(fnp.array(Xf32), fnp.array(IDX), axis=1))["take_along_axis"] / IDX.size))
rows.append(("argsort int64-out (per in elem)", _price(lambda: fnp.argsort(fnp.array(Xf32), axis=1))["argsort"] / Xf32.size))
rows.append(("argpartition bool-in (per in elem)", _price(lambda: fnp.argpartition(fnp.array(MASK), 192, axis=1))["argpartition"] / MASK.size))
rows.append(("einsum nk,nko (per 2nko)", _price(lambda: fnp.einsum("nk,nko->no", fnp.array(Xf32[:, :64]), fnp.array(rng.standard_normal((4096, 64, 256)).astype(np.float32))))["einsum"] / (2 * 4096 * 64 * 256)))
rows.append(("concatenate f32 (per out elem)", _price(lambda: fnp.concatenate([fnp.array(Xf32), fnp.array(Xf32)], axis=0))["concatenate"] / (2 * Xf32.size)))
rows.append(("where f32 (per out elem)", _price(lambda: fnp.where(fnp.array(MASK), fnp.array(Xf32), 0.0))["where"] / Xf32.size))
rows.append(("maximum(x,0) f32 (per elem)", _price(lambda: fnp.maximum(fnp.array(Xf32), 0.0))["maximum"] / Xf32.size))
rows.append(("sum bool axis1 (per in elem)", _price(lambda: fnp.sum(fnp.array(MASK), axis=1))["sum"] / MASK.size))
rows.append(("astype f64->f32 (per elem)", _price(lambda: fnp.array(Wf64).astype(fnp.float32))["astype"] / Wf64.size))
rows.append(("getitem fancy w[idx,:] (per out elem)", _price(lambda: fnp.array(Wf32)[fnp.array(IDX[0]), :])["getitem"] / (64 * 256)))

print(f"{'op (normalization)':42s} {'billed/unit':>11s}")
print("-" * 55)
for name, v in rows:
    print(f"{name:42s} {v:11.3f}")

# pack-vs-dense crossover under f32: packed row of nnz k into out_width o costs
# take(4)*k*o + einsum(2)*k*o (+ order overhead) vs dense 2*in*o*(7/8 strassen).
c_take = rows[3][1]; c_einsum = rows[7][1] * 2
for in_w in (256, 192, 128):
    k_star = 2 * in_w * 0.875 / (c_take + c_einsum)
    print(f"pack beats dense (in_width={in_w}) only for row nnz k < ~{k_star:.0f}")
print("=> old MAX_K=3/4 (k<=192) packs far past the new crossover; idea-5 grid tests ~1/3 and ~1/4")

flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs


op (normalization)                         billed/unit
-------------------------------------------------------
matmul f32@f32 (per 2mkn)                        0.998
matmul f32@f64 (per 2mkn)                        1.996
matmul f64@f64 (per 2mkn)                        1.996
take rows w[order] (per out elem)                4.000
take_along_axis (per out elem)                   4.000
argsort int64-out (per in elem)                 32.000
argpartition bool-in (per in elem)               4.000
einsum nk,nko (per 2nko)                         0.992
concatenate f32 (per out elem)                   1.000
where f32 (per out elem)                         8.000
maximum(x,0) f32 (per elem)                      1.000
sum bool axis1 (per in elem)                     1.992
astype f64->f32 (per elem)                       2.000
getitem fancy w[idx,:] (per out elem)            4.000
pack beats dense (in_width=256) only for row nnz k < ~75
pack beats dense (in_width=192) only for row nnz k < ~56
pack 

flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs
flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.00e+13 FLOPs


## 1. Idea 0 — dtype hygiene: float32 everywhere

**Hypothesis:** casting `mlp.weights` to float32 before any op halves-or-better the billed
FLOPs of the entire surface with no material raw-MSE change (fp32 sampling noise ≪ MC error).

**Prediction:** algo34 baseline (f64 weights) is over budget under v0.9.1 (billed ≈ 408G >
272G → zeroed on grader); the cast brings it back under with multiplier ≈ 0.7–0.8 and
raw MSE within ±0.5%.

**Parameters** — scientific: `_CAST_WEIGHTS_F32`; fixed: everything else at algo34 defaults,
N=61,440, seed 0, 10 nets. **Success:** multiplier < 1 and |ΔMSE| < 2%. **Kill:** MSE
regression > 2%.

In [3]:
results = {}
results["algo34_f64 (ship bytes)"] = run_variant("A_algo34_f64", {}, FULL_IDS)
results["algo34_f32cast"] = run_variant("B_algo34_f32", {"_CAST_WEIGHTS_F32": True}, FULL_IDS)
table(results, baseline="algo34_f64 (ship bytes)")

for name in results:
    ops = results[name][0]["ops"]
    top = sorted(ops.items(), key=lambda kv: -kv[1])[:8]
    tot = sum(ops.values())
    print(f"\n{name} — net 0 billed-FLOP shares:")
    for op, c in top:
        print(f"  {op:16s} {c/1e9:8.2f}G  {100*c/tot:5.1f}%")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
algo34_f64 (ship bytes)       414.9  1.525    10  3.788e-07   1.236e+00  5.701e-07       
algo34_f32cast                209.7  0.771     0  3.789e-07   2.882e-07  2.882e-07 -100.0%

algo34_f64 (ship bytes) — net 0 billed-FLOP shares:
  take               193.48G   47.4%
  matmul             113.94G   27.9%
  einsum              91.17G   22.3%
  add                  2.24G    0.5%
  concatenate          1.62G    0.4%
  argpartition         1.01G    0.2%
  take_along_axis      0.83G    0.2%
  mean                 0.82G    0.2%

algo34_f32cast — net 0 billed-FLOP shares:
  take                97.62G   47.3%
  matmul              56.98G   27.6%
  einsum              45.59G   22.1%
  add                  1.13G    0.5%
  argpartition         1.01G    0.5%
  concatenate          0.81G    0.4%
  mean                

## 2. Idea 1 — dense-only algo34 (315516's routing philosophy on the 317421 surface)

**Hypothesis:** with `take` at 4/elem, the packed gather path costs more than the guarded
2-block Strassen dense path it replaces, so disabling `_PACKED_ROWSPARSE` (all-dense) lowers
billed FLOPs at identical raw MSE (routing is exact).

**Prediction:** dense-only ≥ 15% cheaper billed than packed at f32; N=61,440 fits budget
comfortably; adjusted improves by the same factor. N=40,960 rows trade ~raw-MSE ∝ 1/N against
multiplier ∝ N — the measured bowl decides.

**Parameters** — scientific: `_PACKED_ROWSPARSE`, `_TOTAL_SAMPLES`; fixed: f32 cast ON, algo34
otherwise. **Success:** best adjusted < f32-packed baseline by > 3%. **Kill:** dense-only over
budget or raw MSE shifts (would mean routing is NOT exact).

In [4]:
f32 = {"_CAST_WEIGHTS_F32": True}
results1 = {}
results1["packed N=61440 (=idea0)"] = results["algo34_f32cast"]
results1["dense  N=61440"] = run_variant("C_dense_61440", {**f32, "_PACKED_ROWSPARSE": False}, FULL_IDS)
results1["packed N=40960 (=318620+f32)"] = run_variant("E_packed_40960", {**f32, "_TOTAL_SAMPLES": 40960}, FULL_IDS)
results1["dense  N=40960"] = run_variant("D_dense_40960", {**f32, "_PACKED_ROWSPARSE": False, "_TOTAL_SAMPLES": 40960}, FULL_IDS)
table(results1, baseline="packed N=61440 (=idea0)")

d = np.max(np.abs(results1["dense  N=61440"][0]["pred"][-1] - results1["packed N=61440 (=idea0)"][0]["pred"][-1]))
print(f"\nrouting exactness check (net 0, dense vs packed final row): max|Δ| = {d:.2e} (Strassen fp-noise class)")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
packed N=61440 (=idea0)       209.7  0.771     0  3.789e-07   2.882e-07  2.882e-07       
dense  N=61440                156.7  0.576     0  3.788e-07   2.158e-07  2.158e-07  -25.1%
packed N=40960 (=318620+f32)    139.9  0.515     0  4.295e-07   2.198e-07  2.198e-07  -23.7%
dense  N=40960                104.7  0.385     0  4.295e-07   1.647e-07  1.647e-07  -42.8%

routing exactness check (net 0, dense vs packed final row): max|Δ| = 7.45e-07 (Strassen fp-noise class)


### Idea 1 addendum — where does the fixed-N bowl bottom out now?

Under dense-f32 pricing, billed F ≈ c·N with the 0.1 multiplier floor reached near N ≈ 10.7k.
Since adjusted ≈ (mse∞ + v/N) · max(0.1, cN/B), the optimum should sit at or near the floor —
a very different answer than the old-pricing "bowl is ~4% deep across 14k–61k" result.

**Hypothesis:** adjusted improves monotonically as N shrinks toward the multiplier floor.
**Kill:** raw-MSE inflation at small N (probe/refinement degradation) breaks the trend before
the floor. Note: same Sobol-prefix artifact throughout (no rebuild — realization is
leaderboard-validated); `_BASE_SAMPLES` stays 10,240.

In [5]:
resultsN = {}
for N in (61440, 40960, 30720, 20480, 16384, 12288):
    ov = {**f32, "_PACKED_ROWSPARSE": False}
    tag = {61440: "C_dense_61440", 40960: "D_dense_40960"}.get(N, f"N_dense_{N}")
    if N != 61440:
        ov = {**ov, "_TOTAL_SAMPLES": N}
    resultsN[f"dense N={N}"] = run_variant(tag, ov, FULL_IDS)
table(resultsN, baseline="dense N=61440")
floor_N = int(0.1 * BUDGET / (summ(resultsN["dense N=61440"])["F_G"] * 1e9 / 61440))
print(f"\n0.1-multiplier floor reached near N ≈ {floor_N} at this billing rate")

flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


...

flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


flopscope 0.9.1+np2.2.6 (numpy 2.2.6 backend) | budget: 1.09e+13 FLOPs


variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
dense N=61440                 156.7  0.576     0  3.788e-07   2.158e-07  2.158e-07       
dense N=40960                 104.7  0.385     0  4.295e-07   1.647e-07  1.647e-07  -23.7%
dense N=30720                  78.7  0.289     0  8.215e-07   2.356e-07  2.356e-07   +9.1%
dense N=20480                  52.7  0.194     0  1.476e-06   2.789e-07  2.789e-07  +29.2%
dense N=16384                  42.4  0.156     0  1.966e-06   2.974e-07  2.974e-07  +37.8%
dense N=12288                  32.6  0.120     0  2.754e-06   3.227e-07  3.227e-07  +49.5%

0.1-multiplier floor reached near N ≈ 10663 at this billing rate


## 3. Idea 5 — re-capturing activation sparsity under the new pricing

The starter-kit fact we want back: *the sampled activation matrix is sparse; zeros can be
omitted from the propagating matmuls.* Under v0.9.1 the old gather economy is repriced, so the
question is which (if any) sparse mechanism still beats dense-Strassen.

**Hypothesis:** packing still wins **only** for rows with nnz below the measured crossover
(k* ≈ 60–80 at in_width 256), i.e. deep-layer near-dead rows; the fitted fire-threshold map
(crossover f=0.91) and MAX_K=3/4 over-pack and lose; a recalibrated grid (MAX_K≈1/4–1/3,
lower fire thresholds, layer-1 full-pack off) recovers whatever sparse margin remains.

**Comparison protocol:** routing is exact ⇒ identical predictions; rank by billed FLOPs alone
(6 sweep nets), confirm the winner on 10. **Success:** any sparse config beats dense-only
billed FLOPs. **Kill:** dense-only cheapest everywhere ⇒ sparsity is dead under v0.9.1 and the
ship surface should be dense (this is itself the decision datum).

In [6]:
grid = {
    "dense-only": {**f32, "_PACKED_ROWSPARSE": False},
    "packed baseline (map,3/4)": {**f32},
    "packed MAX_K=1/3": {**f32, "_PACKED_ROWSPARSE_MAX_K_NUM": 1, "_PACKED_ROWSPARSE_MAX_K_DEN": 3},
    "packed MAX_K=1/4": {**f32, "_PACKED_ROWSPARSE_MAX_K_NUM": 1, "_PACKED_ROWSPARSE_MAX_K_DEN": 4},
    "packed fire=0.5 uniform": {**f32, "_BLOCK_SPLIT_FIRE_THRESH": 0.5, "_BLOCK_SPLIT_FIRE_THRESH_BY_LAYER": {}},
    "packed L1 split (not full-pack)": {**f32, "_LAYER1_FULL_PACK": False},
    "packed 1/4 + L1split + fire.5": {**f32, "_PACKED_ROWSPARSE_MAX_K_NUM": 1, "_PACKED_ROWSPARSE_MAX_K_DEN": 4,
                                       "_LAYER1_FULL_PACK": False, "_BLOCK_SPLIT_FIRE_THRESH": 0.5,
                                       "_BLOCK_SPLIT_FIRE_THRESH_BY_LAYER": {}},
}
results5 = {}
for name, ov in grid.items():
    tag = "R5_" + hashlib.md5(name.encode()).hexdigest()[:8]
    results5[name] = run_variant(tag, ov, SWEEP_IDS)
table(results5, baseline="dense-only")

ref = results5["dense-only"]
for name, rows in results5.items():
    dmax = max(np.max(np.abs(r["pred"][-1] - ref[k]["pred"][-1])) for k, r in enumerate(rows))
    print(f"{name:34s} max final-row |Δ| vs dense = {dmax:.2e}")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
dense-only                    158.5  0.583     0  4.532e-07   2.602e-07  2.602e-07       
packed baseline (map,3/4)     212.6  0.782     0  4.533e-07   3.476e-07  3.476e-07  +33.6%
packed MAX_K=1/3              170.3  0.626     0  4.533e-07   2.801e-07  2.801e-07   +7.6%
packed MAX_K=1/4              170.2  0.626     0  4.533e-07   2.797e-07  2.797e-07   +7.5%
packed fire=0.5 uniform       164.2  0.604     0  4.533e-07   2.705e-07  2.705e-07   +4.0%
packed L1 split (not full-pack)    212.6  0.782     0  4.533e-07   3.477e-07  3.477e-07  +33.6%
packed 1/4 + L1split + fire.5    158.8  0.584     0  4.533e-07   2.614e-07  2.614e-07   +0.5%
dense-only                         max final-row |Δ| vs dense = 0.00e+00
packed baseline (map,3/4)          max final-row |Δ| vs dense = 1.43e-06
packed MAX_K=1/3            

## 4. Idea 2 — the layer-30/31 fold and earlier identity treatment of on-neurons

Memory context: the 2026-07-09 result (matmul-efficiency-tapped) showed the on-neuron linear
fold across depth is NOT a FLOP saving — dense fan-out costs an extra full matmul per layer;
the terminal 30/31 fold exploits the only free case. Matmul pricing didn't change shape in
v0.9.1 (only dtype rates), so the FLOP side of that result carries over. What we CAN tune:

- **fold off** — is the fold still net-positive at all under new pricing?
- **identity threading** (layers L..29 skip ReLU on on-columns) — a pure accuracy probe of
  early linearization: if MSE is flat down to L≈25, the approximation is safe and any future
  FLOP mechanism can use it; if it degrades, earlier folding is dead regardless of FLOPs.
- **on-threshold** (what counts as "on" at 30/31) — lower = more neurons take the linear path
  (smaller sampled kink block at the fold), higher = more exact sampling.

**Hypothesis:** fold stays positive; identity threading is MSE-neutral to L≈27 then degrades;
on-threshold 3.0 is near-optimal (the bowl is shallow). **Success:** any variant beats
dense-only adjusted by > 2% (else CLOSE this direction). All on dense-only f32, N=61,440,
6 nets.

In [7]:
dense = {**f32, "_PACKED_ROWSPARSE": False}
grid2 = {
    "fold on (baseline)": dense,
    "fold OFF": {**dense, "_FOLD_ENABLED": False, "_REFINE_DEAD_STOP_LAYER": 30, "_DEMOTE_ACTIVE_DEAD_STOP_LAYER": 30},
    "identity from L=29": {**dense, "_IDENTITY_ON_START_LAYER": 29},
    "identity from L=27": {**dense, "_IDENTITY_ON_START_LAYER": 27},
    "identity from L=25": {**dense, "_IDENTITY_ON_START_LAYER": 25},
    "on-thresh 2.5": {**dense, "_ON_THRESH": 2.5, "_PILOT_ON_THRESH": 2.5, "_ON_PROBE_MAX": 3.5},
    "on-thresh 3.5": {**dense, "_ON_THRESH": 3.5, "_PILOT_ON_THRESH": 3.5, "_ON_PROBE_MAX": 4.5},
}
results2 = {}
for name, ov in grid2.items():
    tag = "I2_" + hashlib.md5(name.encode()).hexdigest()[:8]
    results2[name] = run_variant(tag, ov, SWEEP_IDS)
table(results2, baseline="fold on (baseline)")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
fold on (baseline)            158.5  0.583     0  4.532e-07   2.602e-07  2.602e-07       
fold OFF                      160.0  0.588     0  4.530e-07   2.631e-07  2.631e-07   +1.1%
identity from L=29            158.5  0.583     0  4.556e-07   2.616e-07  2.616e-07   +0.5%
identity from L=27            158.5  0.583     0  4.586e-07   2.633e-07  2.633e-07   +1.2%
identity from L=25            158.5  0.583     0  4.611e-07   2.648e-07  2.648e-07   +1.7%
on-thresh 2.5                 157.9  0.581     0  4.606e-07   2.636e-07  2.636e-07   +1.3%
on-thresh 3.5                 159.0  0.584     0  4.521e-07   2.602e-07  2.602e-07   +0.0%


## 5. Idea 3 — skip the 5% pilot stage (analytic-primary, single 20% recheck)

The analytic classification already restricts probe candidates to the borderline windows
(dead: α∈[−4,−3), on: α∈(3,4], demote: α∈(−3,−2.5]). The staged pilot then spends a 5% pass on
ALL candidates and a 20% pass on the uncertain subset.

**Hypothesis:** classifying every candidate directly with the 20% sampled alpha (no 5% stage)
is at least accuracy-neutral (every decision uses 4× more probe rows) and FLOP-neutral-or-better
(one pass over all candidates vs two passes with overlap).

**Prediction:** raw MSE −0.5%..+0.5% (classification changes a few near-threshold neurons),
billed FLOPs ±0.3%. **Success:** adjusted not worse; simpler code. **Kill:** MSE > +1%.

In [8]:
results3 = {}
results3["staged 5%+20% (baseline)"] = results2["fold on (baseline)"]
results3["single 20% stage"] = run_variant("I3_single20", {**dense, "_PILOT_MODE": "single20"}, SWEEP_IDS)
table(results3, baseline="staged 5%+20% (baseline)")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
staged 5%+20% (baseline)      158.5  0.583     0  4.532e-07   2.602e-07  2.602e-07       
single 20% stage              159.0  0.584     0  4.532e-07   2.611e-07  2.611e-07   +0.3%


## 6. Idea 4 — dead-neuron reporting on the scored row

Final-layer dead neurons (α<−3, never sampled) currently get their analytic ADF mean. The truth
for such cells is a small positive value; candidates: zero, analytic, or a scaled blend γ.

**Hypothesis:** analytic (γ=1) beats zero (γ=0) because E[relu(z)] > 0 always; the optimal γ is
near 1 unless the deep-layer ADF mean is systematically biased (replicate-bias memory: fleet
bias ≈ 1.4e-8, so expect a shallow bowl near 1). The γ sweep is post-hoc on captured
components — one run, no re-predicts. **Decision:** ship the argmin γ only if it beats γ=1 by
more than the seed-noise scale, else keep analytic.

In [9]:
cap_rows = run_variant("C_dense_61440", {**f32, "_PACKED_ROWSPARSE": False}, FULL_IDS, capture=True)
gammas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
mses = {g: [] for g in gammas}
dead_stats = []
for r in cap_rows:
    fin = r["cap"]["final_row"]; corr = r["cap"]["dead_corr_final"]
    truth = GT[r["net"]][-1]
    dead_idx = r["cap"]["dead_idx_final"]
    for g in gammas:
        mses[g].append(float(((fin - (1 - g) * corr - truth) ** 2).mean()))
    if len(dead_idx):
        dead_stats.append((len(dead_idx), float(truth[dead_idx].mean()), float(corr[dead_idx].mean()),
                           float(((corr[dead_idx] - truth[dead_idx]) ** 2).sum() / (256 * ((fin - truth) ** 2).mean()))))

print(f"{'gamma':>6s} {'mean final MSE':>14s} {'Δ vs γ=1':>9s}")
base_mse = np.mean(mses[1.0])
for g in gammas:
    m = np.mean(mses[g])
    print(f"{g:6.2f} {m:14.4e} {100*(m/base_mse-1):+8.2f}%")
nd = np.mean([s[0] for s in dead_stats])
print(f"\ndead cells/net ≈ {nd:.0f}; mean truth {np.mean([s[1] for s in dead_stats]):.3e} "
      f"vs analytic {np.mean([s[2] for s in dead_stats]):.3e}; "
      f"dead share of final SSE ≈ {100*np.mean([s[3] for s in dead_stats]):.2f}%")

 gamma mean final MSE  Δ vs γ=1
  0.00     3.7899e-07    +0.04%
  0.25     3.7894e-07    +0.03%
  0.50     3.7890e-07    +0.02%
  0.75     3.7887e-07    +0.01%
  1.00     3.7884e-07    +0.00%
  1.25     3.7882e-07    -0.01%
  1.50     3.7881e-07    -0.01%

dead cells/net ≈ 80; mean truth 1.125e-05 vs analytic 4.214e-06; dead share of final SSE ≈ 0.88%


## 7. Summary, decision, and bench log

In [10]:
final = {}
final["algo34 ship bytes (f64, packed, 61440)"] = results["algo34_f64 (ship bytes)"]
final["+ f32 cast"] = results["algo34_f32cast"]
final["f32 dense-only 61440  «idea 1»"] = results1["dense  N=61440"]
final["f32 dense-only 40960"] = results1["dense  N=40960"]
final["f32 packed 40960 (318620+cast)"] = results1["packed N=40960 (=318620+f32)"]
for name, rows in resultsN.items():
    if name not in ("dense N=61440", "dense N=40960"):
        final[f"f32 dense-only {name.split('=')[1]}"] = rows
table(final, baseline="algo34 ship bytes (f64, packed, 61440)")

best = min(final, key=lambda k: summ(final[k])["adj"])
print(f"\nbest on 10 nets: {best}")
print("Reminder: 315516 (current leaderboard best) is the July-9 adaptive-N gather-free surface;")
print("it is NOT replicated here — the reference comparison is on the grader, not local.")

variant                        F(G)   mult #over    raw MSE adj(grader) adj(ideal)   Δadj%
------------------------------------------------------------------------------------------
algo34 ship bytes (f64, packed, 61440)    414.9  1.525    10  3.788e-07   1.236e+00  5.701e-07       
+ f32 cast                    209.7  0.771     0  3.789e-07   2.882e-07  2.882e-07 -100.0%
f32 dense-only 61440  «idea 1»    156.7  0.576     0  3.788e-07   2.158e-07  2.158e-07 -100.0%
f32 dense-only 40960          104.7  0.385     0  4.295e-07   1.647e-07  1.647e-07 -100.0%
f32 packed 40960 (318620+cast)    139.9  0.515     0  4.295e-07   2.198e-07  2.198e-07 -100.0%
f32 dense-only 30720           78.7  0.289     0  8.215e-07   2.356e-07  2.356e-07 -100.0%
f32 dense-only 20480           52.7  0.194     0  1.476e-06   2.789e-07  2.789e-07 -100.0%
f32 dense-only 16384           42.4  0.156     0  1.966e-06   2.974e-07  2.974e-07 -100.0%
f32 dense-only 12288           32.6  0.120     0  2.754e-06   3.227e-07

In [11]:
from datetime import date
log = REPO / "bench_logs" / "submission_learnings_2026-07-25.md"
lines = ["# Submission Learnings - 2026-07-25", "",
         "## PR150/v0.9.1 local experiment notebook (experiments_pr150.ipynb)", "",
         f"- flopscope v0.9.1 (merged #150+#151) from `.flopscope-pr150/`; budget {BUDGET/1e9:.0f}G;",
         f"  {len(FULL_IDS)} mini nets {FULL_IDS}; FLOP-only multiplier, grader-true zeroing.", ""]
for name, rows in final.items():
    s = summ(rows)
    lines.append(f"- {name}: F={s['F_G']:.1f}G mult={s['mult']:.3f} over={s['n_over']} "
                 f"raw={s['raw']:.3e} adj={s['adj']:.3e}")
lines += ["", "### Sweeps (6 nets)", ""]
for title, res in [("idea1 fixed-N sweep (10 nets)", "resultsN"), ("idea5 routing", "results5"),
                   ("idea2 fold/identity/threshold", "results2"), ("idea3 pilot", "results3")]:
    lines.append(f"**{title}**")
    for name, rows in eval(res).items():
        s = summ(rows)
        lines.append(f"- {name}: F={s['F_G']:.1f}G adj={s['adj']:.3e} raw={s['raw']:.3e}")
    lines.append("")
log.write_text("\n".join(lines) + "\n")
print(f"wrote {log}")

wrote /Users/natashastewart/whest-kprop-experiments/bench_logs/submission_learnings_2026-07-25.md


### Grader-parity caveats and next steps

- This harness is standalone in-process; the grader runs the subprocess evaluator with λ·R
  (residual wall) added to effective compute. Gather-free dense paths also *reduce* wall time,
  so the FLOP-only ranking here is conservative for dense variants.
- Local flopscope in `.venv` is still stock 0.8.0rc5 — `whest run` there does NOT reflect
  v0.9.1 pricing. Re-validate any ship candidate with the `.flopscope-pr150` build on the
  path, then package (FOLDER, not file), `tar -tzf`, and **stop for approval before
  `whest submit`** (standing gate).
- Next: promote the winning config into a submission folder derived from the verified 317421
  bytes + the winning diffs, and grade one probe submission.